In [1]:
2+2

4

In [2]:
%pip install -q \
    "langchain>=0.2.0,<0.3.0" \
    "langchain-core>=0.2.0,<0.3.0" \
    "langchain-community>=0.2.0,<0.3.0" \
    "langchain-google-genai" \
    "faiss-cpu>=1.9.0" \
    "sentence-transformers" \
    "ragas" \
    "datasets" \
    "langchain-text-splitters"

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + c:\Users\vaish\agenticaisep\.venv13\Scripts\python.exe C:\Users\vaish\AppData\Local\Temp\pip-install-ucq9fbti\numpy_e725e35fb2af4fe6a466656dfccd1ee2\vendored-meson\meson\meson.py setup C:\Users\vaish\AppData\Local\Temp\pip-install-ucq9fbti\numpy_e725e35fb2af4fe6a466656dfccd1ee2 C:\Users\vaish\AppData\Local\Temp\pip-install-ucq9fbti\numpy_e725e35fb2af4fe6a466656dfccd1ee2\.mesonpy-wbb_qf4p -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\vaish\AppData\Local\Temp\pip-install-ucq9fbti\numpy_e725e35fb2af4fe6a466656dfccd1ee2\.mesonpy-wbb_qf4p\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\vaish\AppData\Local\Temp\pip-install-ucq9fbti\numpy_e725e35fb2af4fe6a466656dfccd1ee2
      Build dir: C:\Users\vaish\AppData\Local\Temp\pip-install-ucq9fbti\numpy

In [15]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from dotenv import  load_dotenv

load_dotenv()

api_key=os.getenv("GEMINI_API_KAY")


In [16]:
llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.0
)

embedder=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

judge_llm=llm

In [17]:
from langchain_core.documents import  Document
from langchain_community.vectorstores import FAISS

audit_data = [
Document(page_content="MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate.", metadata={"id": 1}),
Document(page_content="Tuesday Review confirmed the 15% spike is due to 'Legacy Scan-X' hardware and firmware v2.1.", metadata={"id": 2}),
Document(page_content="Jaymin approved a $45,000 emergency budget to upgrade European scanners by Q1 end.", metadata={"id": 3}),
Document(page_content="OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.", metadata={"id": 4}),
Document(page_content="Tony recommends a distributed architecture for handling 500+ PDFs in legacy branches.", metadata={"id": 5}),
Document(page_content="The 15% error rate is classified as 'Critical' for Banking and Compliance audits.", metadata={"id": 6}),
Document(page_content="Marten's team is monitoring OCR logs 24/7 until the hardware upgrade is finished.", metadata={"id": 7}),
Document(page_content="European legacy branches are the only units still using the v2.1 firmware.", metadata={"id": 8}),
Document(page_content="The new firmware v3.0 has been successfully tested in the North American cluster.", metadata={"id": 9}),
Document(page_content="Budget allocation for Q1 also includes a 10% reserve for unexpected cloud egress costs.", metadata={"id": 10}),
Document(page_content="Anisha suggested moving OCR processing to an asynchronous queue using RabbitMQ.", metadata={"id": 11}),
Document(page_content="Legacy Scan-X machines have a known overheating issue when processing over 100 pages.", metadata={"id": 12}),
Document(page_content="Compliance team noted that OCR errors are leading to incorrect data in customer KYC files.", metadata={"id": 13}),
Document(page_content="The upgrade project is codenamed 'Project Vision' and is led by the MLOps Core team.", metadata={"id": 14}),
Document(page_content="Handwritten PDF recognition accuracy dropped to 62% in the last batch test.", metadata={"id": 15}),
Document(page_content="Security audit found that legacy firmware v2.1 has three unpatched vulnerabilities.", metadata={"id": 16}),
Document(page_content="Training data for the new OCR model includes 50,000 samples of handwritten European scripts.", metadata={"id": 17}),
Document(page_content="The hardware vendor 'OptiScan' has been notified about the hardware failures.", metadata={"id": 18}),
Document(page_content="A temporary patch was deployed on Monday to reduce memory leaks during batch processing.", metadata={"id": 19}),
Document(page_content="Q2 Roadmap: Complete migration of all legacy branches to the centralized MLOps platform.", metadata={"id": 20})

]


In [18]:
vectorstore=FAISS.from_documents(audit_data,embedder)
base_retriever=vectorstore.as_retriever(search_kwargs={"k":20})

In [19]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


# 1 .Initialize the model

cross_encoder_model=HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

#2.Configure the re ranker

reranker= CrossEncoderReranker(model=cross_encoder_model,top_n=3)

#3.BUILD 2nd stage retriever
compression_retriever=ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3611.07it/s]


In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template="""Answer the question bsed oly on the following context

{context}

question:{question}
Answer: """

prompt=ChatPromptTemplate.from_template(template)


advanced_rerank_chain=(
    {"context":compression_retriever,"question":lambda x:x}
    |prompt
    |llm
    |StrOutputParser()
)


In [21]:
golden_dataset = [

{

"question": "What is the OCR failure rate in European legacy branches?",
"ground_truth": "The OCR failure rate is 15%."

},

{

"question": "Why do OCR failures peak on Tuesdays?",
"ground_truth": "Due to weekly bulk-batch processing of handwritten PDFs."

},

{

"question": "What is the codename for the upgrade project?",
"ground_truth": "The project codename is 'Project Vision'."

}

]

In [29]:
evaluation_data={ "user_input":[],"response":[],"retrieved_contexts":[],"reference":[]}


In [30]:
for item in golden_dataset:
    query=item["question"]

    retrieved_docs=compression_retriever.invoke(query)
    context_list=[doc.page_content for doc in retrieved_docs]
    response=advanced_rerank_chain.invoke(query)

    evaluation_data["user_input"].append(query)
    evaluation_data["retrieved_contexts"].append(context_list)
    evaluation_data["response"].append(response)
    evaluation_data["reference"].append(item["ground_truth"])

print(f"Data generation complete: {len(evaluation_data["user_input"])} samples")

c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Data generation complete: 3 samples


In [ ]:
%pip install datasets

In [ ]:
%pip install ragas

In [ ]:
%pip show ragas
%pip show langchain
%pip show langchain-community
%pip show langchain-core

In [31]:
import sys
from unittest.mock import MagicMock
# Fix dependency issue: shim removed legacy vertexai module for ragas compatibility
sys.modules.setdefault('langchain_community.chat_models.vertexai', MagicMock())

from datasets import Dataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


In [32]:
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    AnswerCorrectness
)

C:\Users\vaish\AppData\Local\Temp\ipykernel_25792\2201604522.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\vaish\AppData\Local\Temp\ipykernel_25792\2201604522.py:1: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\vaish\AppData\Local\Temp\ipykernel_25792\2201604522.py:1: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\vaish\AppData\Local\Temp\ipykernel_25792\

In [33]:
wrapped_llm = LangchainLLMWrapper(judge_llm)
wrapped_embedder = LangchainEmbeddingsWrapper(embedder)

faithfulness_obj = Faithfulness()
answer_relevancy_obj = AnswerRelevancy()
context_precision_obj = ContextPrecision()
context_recall_obj = ContextRecall()
answer_correctness_obj = AnswerCorrectness()

C:\Users\vaish\AppData\Local\Temp\ipykernel_25792\3104243326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  wrapped_llm = LangchainLLMWrapper(judge_llm)
C:\Users\vaish\AppData\Local\Temp\ipykernel_25792\3104243326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  wrapped_embedder = LangchainEmbeddingsWrapper(embedder)


In [34]:
import pandas as pd

if len(evaluation_data['user_input']) > 0:

    ragas_dataset = Dataset.from_dict(evaluation_data)
    results = evaluate(
    dataset = ragas_dataset,
    metrics=[
    faithfulness_obj,
    answer_relevancy_obj,
    context_precision_obj,
    context_recall_obj,
    answer_correctness_obj
    ],

    llm = wrapped_llm,
    embeddings= wrapped_embedder
    )

    results_df = results.to_pandas()
    display(results_df)

else:

    print("Error: Evaluation data is Empty")

Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.
c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generat

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness
0,What is the OCR failure rate in European legac...,[MLOps Audit Q4: European division legacy bran...,"Based on the provided context, the OCR failure...",The OCR failure rate is 15%.,1.0,0.948398,1.0,1.0,0.975590
1,Why do OCR failures peak on Tuesdays?,[OCR failures peak on Tuesdays due to weekly b...,"Based on the provided context, OCR failures pe...",Due to weekly bulk-batch processing of handwri...,1.0,0.881266,1.0,1.0,0.721270
2,What is the codename for the upgrade project?,[The upgrade project is codenamed 'Project Vis...,Project Vision,The project codename is 'Project Vision'.,1.0,0.730259,1.0,1.0,0.971493
